# GBP/USD Cleaning Raw Data

In [2]:
import pandas as pd
import numpy as np

## Load Data

In [3]:
gbp_usd_raw = pd.read_csv("../data/raw/gbpusd/v1_11-09-25/DEXUSUK.csv", parse_dates=["observation_date"], index_col="observation_date")

# The Federal Reserve’s “noon buying rate”, which is a daily reference rate.
# The price at which the Fed would buy foreign currency from cable transfers payable in foreign currencies — basically an official spot rate quotation.

In [4]:
gbp_usd_raw.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 14270 entries, 1971-01-04 to 2025-09-12
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   DEXUSUK  13717 non-null  float64
dtypes: float64(1)
memory usage: 223.0 KB


In [5]:
gbp_usd_raw.head()

,DEXUSUK
observation_date,
1971-01-04,2.3938
1971-01-05,2.3949
1971-01-06,2.3967
1971-01-07,2.3963
1971-01-08,2.3972


## Clean

In [6]:
gbp_usd_cleaned = gbp_usd_raw.copy()

### Column Structuring

In [7]:
gbp_usd_cleaned.index = gbp_usd_raw.index.rename("date")

In [8]:
gbp_usd_cleaned = gbp_usd_cleaned.rename(columns={"DEXUSUK": "gbp_usd_noon_rate"})

### Handling Null Values

In [9]:
gbp_usd_cleaned["gbp_usd_noon_rate"].isna().sum()

np.int64(553)

In [10]:
expected_dates = pd.date_range(start=gbp_usd_cleaned.index.min(), end=gbp_usd_cleaned.index.max(), freq="B")
missing_d = gbp_usd_cleaned.index.difference(expected_dates)
missing_d.shape[0]

0

In [11]:
good_data = gbp_usd_cleaned[gbp_usd_cleaned["gbp_usd_noon_rate"].notna()]

In [12]:
good_data.index.diff().value_counts()

date
1 days     10694
3 days      2479
4 days       366
2 days       170
5 days         6
10 days        1
Name: count, dtype: int64

In [13]:
end_of_missing_period = good_data[good_data.index.diff() == "10 days"].index[0]

missing_period = gbp_usd_cleaned.loc[end_of_missing_period - pd.Timedelta(days=9) : end_of_missing_period - pd.Timedelta(days=1)]

In [14]:
gbp_usd_cleaned["gbp_usd_noon_rate"] = gbp_usd_cleaned["gbp_usd_noon_rate"].ffill()

In [15]:
gbp_usd_cleaned["gbp_usd_noon_rate"].isna().sum()

np.int64(0)

In [16]:
gbp_usd_cleaned.loc[end_of_missing_period - pd.Timedelta(days=9) : end_of_missing_period - pd.Timedelta(days=1)]

,gbp_usd_noon_rate
date,
1971-08-16,2.4195
1971-08-17,2.4195
1971-08-18,2.4195
1971-08-19,2.4195
1971-08-20,2.4195


In [17]:
gbp_usd_cleaned.loc[end_of_missing_period - pd.Timedelta(days=9) : end_of_missing_period - pd.Timedelta(days=1)] = np.nan

In [18]:
gbp_usd_cleaned.loc[end_of_missing_period - pd.Timedelta(days=10) : end_of_missing_period]

,gbp_usd_noon_rate
date,
1971-08-13,2.4195
1971-08-16,NaN
1971-08-17,NaN
1971-08-18,NaN
1971-08-19,NaN
1971-08-20,NaN
1971-08-23,2.4350


### Validate Time-Series

In [19]:
assert gbp_usd_cleaned.index.is_monotonic_increasing

## Export

In [21]:
gbp_usd_cleaned.to_csv("../data/cleaned/gbpusd/v1_11-09-25/gbpusd_cleaned_as_of_110925.csv")
gbp_usd_cleaned.dtypes.to_csv("../data/cleaned/gbpusd/v1_11-09-25/gbpusd_cleaned_dtypes_as_of_110925.csv")